# سبق 11 - ایجنٹ سے ایجنٹ (A2A) پروٹوکول


## سیٹ اپ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A پروٹوکول کیا ہے؟

The **ایجنٹ-ٹو-ایجنٹ (A2A) پروٹوکول** ایک اوپن اسٹینڈرڈ ہے جو AI ایجنٹس کو بات چیت کرنے،
ایک دوسرے کو دریافت کرنے، اور باہمی تعاون کرنے کے قابل بناتا ہے — چاہے وہ مختلف فریم ورکس پر بنے ہوں یا مختلف سروسز پر ہوسٹ کیے گئے ہوں۔

Key concepts:

- **کھوج** – ایجنٹس ایک *ایجنٹ کارڈ* شائع کرتے ہیں جو ان کی صلاحیتوں کو بیان کرتا ہے، جس سے
  دوسرے ایجنٹس (یا آرکیسٹریٹرز) کے لیے کسی کام کے لیے درست ماہر تلاش کرنا آسان ہوجاتا ہے۔
- **Message Passing** – ایجنٹس ایک مشترکہ پروٹوکول کے ذریعے منظم پیغامات کا تبادلہ کرتے ہیں، تاکہ ایک
  ایجنٹ کی جانب سے کی گئی درخواست کو دوسرے ایجنٹ سمجھ سکے اور پورا کر سکے چاہے اس کی داخلی
  implementation مختلف کیوں نہ ہو۔
- **Task Lifecycle** – A2A ایسے حالات کی وضاحت کرتا ہے جیسے *submitted*, *working*, *completed*, اور
  *failed*, جس سے آرکیسٹریٹر کو یہ مکمل بصیرت ملتی ہے کہ سپرد کی گئی کام کس طرح پیش رفت کر رہا ہے۔

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## خصوصی سفری ایجنٹس تیار کرنا


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ورک فلو کے ذریعے متعدد ایجنٹس کا تعاون

ہم تینوں ایجنٹس کو ایک ترتیب وار ورک فلو میں جوڑتے ہیں جو A2A پیغام رسانی کی عکاسی کرتا ہے:

1. **CurrencyExchangeAgent** صارف کی درخواست وصول کرتا ہے اور کرنسی سے متعلق رہنمائی فراہم کرتا ہے۔
2. **ActivityPlannerAgent** غنی شدہ سیاق و سباق وصول کرتا ہے اور سرگرمیوں کی سفارشات شامل کرتا ہے۔
3. **TravelManagerAgent** دونوں ان پٹس کو یکجا کرکے ایک حتمی سفری خلاصہ تیار کرتا ہے۔


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## پروڈکشن میں A2A کو سمجھنا

پروڈکشن ماحول میں A2A پروٹوکول طاقتور کراس سروس منظرنامے فعال کرتا ہے:

| صلاحیت | تفصیل |
|---|---|
| **فریم ورک کے درمیان انٹرآپریبلٹی** | ایک ایجنٹ جو ایک فریم ورک کے ساتھ بنایا گیا ہو دوسرے کسی بھی A2A مطابقت رکھنے والے فریم ورک کے ساتھ بنے ایجنٹ کو کام سونپ سکتا ہے، جس سے واقعی تنظیموں کے درمیان باہمی تعاون ممکن ہوتا ہے۔ |
| **سروس کی حدود** | ایجنٹس الگ مائیکرو سروسز، کلاؤڈ ریجنز، یا حتیٰ کہ مختلف تنظیموں میں ہوسکتے ہیں، اور پھر بھی بغیر کسی رکاوٹ کے مل کر کام کر سکتے ہیں۔ |
| **متغیر دریافت** | ایک آرکیسٹریٹر رن ٹائم میں ایجنٹ کارڈ رجسٹری سے استعلام کر سکتا ہے تاکہ کسی مخصوص ذیلی کام کے لیے بہترین ماہر تلاش کیا جا سکے۔ |
| **اسٹریمنگ اور پش نوٹیفیکیشنز** | A2A حقیقی وقت کی پیشرفت اپڈیٹس کے لیے Server-Sent Events (SSE) کی حمایت کرتا ہے اور طویل دورانیے کے کاموں کے لیے پش نوٹیفیکیشنز فراہم کرتا ہے۔ |

اوپر جو ورک فلو ہم نے بنایا ہے وہ اس پیٹرن کا ایک سادہ، ان-پروسیس ورژن ہے۔ حقیقی
تعیناتی میں ہر ایجنٹ ایک HTTP endpoint ظاہر کرے گا، ایک Agent Card شائع کرے گا، اور بات چیت
A2A JSON-RPC پروٹوکول کے ذریعے۔


## خلاصہ

اس سبق میں آپ نے سیکھا:

1. **A2A پروٹوکول کیا ہے** — ایجنٹ سے ایجنٹ تک دریافت، پیغام رسانی,
   اور ٹاسک مینجمنٹ.
2. **خصوصی ایجنٹس کیسے بنائیں** — ایک کرنسی ایکسچینج ایجنٹ، ایک سرگرمی پلانر ایجنٹ، اور ایک ٹریول مینیجر آرکیسٹریٹر.
3. **ایجنٹس کو ورک فلو میں مربوط کرنے کا طریقہ** — `WorkflowBuilder` کا استعمال ترتیب وار
   ایجنٹس کے درمیان پیغام رسانی کی ماڈلنگ کے لیے.
4. **A2A پروڈکشن میں کیسے کام کرتا ہے** — متحرک دریافت اور اسٹریمنگ اپڈیٹس کے ساتھ کراس-فریم ورک، کراس-سروس تعاون کو ممکن بنانا.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ردِ ذمہ داری:
اس دستاویز کا ترجمہ AI ترجمہ سروس Co-op Translator (https://github.com/Azure/co-op-translator) کے ذریعے کیا گیا ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، براہِ کرم نوٹ کریں کہ خودکار تراجم میں غلطیاں یا بے ضابطگیاں ہو سکتی ہیں۔ اصل دستاویز کو اس کی مادری زبان میں موجود متن ہی مستند ماخذ سمجھا جانا چاہیے۔ اہم معلومات کے لیے پیشہ ور انسانی مترجم کی خدمات تجویز کی جاتی ہیں۔ ہم اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تعبیر کے لیے ذمہ دار نہیں ہیں۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
